# 07 Political-Corruption Attention Over Time

This notebook loads the cleaned corruption-query denominator tables and the full-corpus political-corruption classifier outputs. It builds weekly and monthly country-level attention measures and plots relative attention over time.

Important: the denominator is the cleaned corruption-query corpus, not all news coverage. The relative-attention measure is therefore the share of corruption-query articles classified as primarily about political corruption.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

PIPELINE_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "political_corruption_pipeline"
)
CLASSIFIER_DIR = PIPELINE_DIR / "silver_classifier"
CLASSIFIED_DIR = CLASSIFIER_DIR / "classified_country_files"
FIGURE_DIR = PIPELINE_DIR / "attention_figures"
TABLE_DIR = PIPELINE_DIR / "attention_tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_ORDER = [
    "Bulgaria",
    "France",
    "Hungary",
    "Italy",
    "Netherlands",
    "Serbia",
    "Sweden",
    "Ukraine",
    "United_Kingdom",
]

COUNTRY_LABELS = {
    "United_Kingdom": "United Kingdom",
}

# Monthly is usually clearer for manuscript figures; weekly is also computed below.
PLOT_LEVEL = "month"  # "month" or "week"

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Pipeline dir: {PIPELINE_DIR}")
print(f"Classified dir: {CLASSIFIED_DIR}")

## 2. Load Denominator Tables

These are the total cleaned corruption-query articles by country-week and country-month.

In [ ]:
denom_week = pd.read_csv(PIPELINE_DIR / "denominator_country_week.csv")
denom_month = pd.read_csv(PIPELINE_DIR / "denominator_country_month.csv")

denom_week["week"] = pd.to_datetime(denom_week["week"])
denom_month["month"] = pd.to_datetime(denom_month["month"])

denom_week = denom_week.rename(columns={"total_articles": "total_coverage"})
denom_month = denom_month.rename(columns={"total_articles": "total_coverage"})

denom_week["country"] = pd.Categorical(denom_week["country"], COUNTRY_ORDER, ordered=True)
denom_month["country"] = pd.Categorical(denom_month["country"], COUNTRY_ORDER, ordered=True)

print("Weekly denominator:", denom_week.shape)
display(denom_week.head())
print("Monthly denominator:", denom_month.shape)
display(denom_month.head())

## 3. Load Classified Political-Corruption Articles

This reads the country-level classifier outputs and keeps only the date, country, and prediction columns needed for aggregation.

In [ ]:
use_columns = {"country", "date_parsed", "year", "month", "week", "pred_political_corruption"}
frames = []

for country in COUNTRY_ORDER:
    path = CLASSIFIED_DIR / f"{country}_classified.csv.gz"
    if not path.exists():
        print(f"Missing classified file: {path}")
        continue

    data = pd.read_csv(path, usecols=lambda column: column in use_columns)
    data["country"] = country
    data = data[data["pred_political_corruption"].eq(1)].copy()
    frames.append(data)
    print(f"Loaded {country}: {len(data):,} political-corruption articles")

if not frames:
    raise FileNotFoundError(f"No classified files found in {CLASSIFIED_DIR}")

pc_articles = pd.concat(frames, ignore_index=True)
pc_articles["date_parsed"] = pd.to_datetime(pc_articles["date_parsed"], errors="coerce", utc=True)
pc_articles["date_naive"] = pc_articles["date_parsed"].dt.tz_convert(None)

if "month" in pc_articles.columns:
    pc_articles["month"] = pd.to_datetime(pc_articles["month"], errors="coerce")
else:
    pc_articles["month"] = pc_articles["date_naive"].dt.to_period("M").dt.to_timestamp()

if "week" in pc_articles.columns:
    pc_articles["week"] = pd.to_datetime(pc_articles["week"], errors="coerce")
else:
    pc_articles["week"] = pc_articles["date_naive"].dt.to_period("W").dt.start_time

pc_articles["country"] = pd.Categorical(pc_articles["country"], COUNTRY_ORDER, ordered=True)

print(f"Total political-corruption articles loaded: {len(pc_articles):,}")
display(pc_articles.head())

## 4. Build Weekly And Monthly Attention Tables

`relative_attention` is the percentage of cleaned corruption-query articles classified as political corruption in a country-period.

In [ ]:
pc_week = (
    pc_articles.groupby(["country", "week"], observed=True)
    .size()
    .reset_index(name="political_corruption_articles")
)

pc_month = (
    pc_articles.groupby(["country", "month"], observed=True)
    .size()
    .reset_index(name="political_corruption_articles")
)

attention_week = denom_week.merge(pc_week, on=["country", "week"], how="left")
attention_month = denom_month.merge(pc_month, on=["country", "month"], how="left")

for data in [attention_week, attention_month]:
    data["political_corruption_articles"] = data["political_corruption_articles"].fillna(0).astype(int)
    data["relative_attention"] = data["political_corruption_articles"] / data["total_coverage"]
    data["relative_attention_pct"] = data["relative_attention"] * 100
    data["country_label"] = data["country"].astype(str).replace(COUNTRY_LABELS)

attention_week.to_csv(TABLE_DIR / "political_corruption_attention_week.csv", index=False)
attention_month.to_csv(TABLE_DIR / "political_corruption_attention_month.csv", index=False)

print(f"Saved attention tables to: {TABLE_DIR}")
display(attention_month.head())

## 5. Overall Summary

In [ ]:
country_summary = (
    attention_month.groupby(["country", "country_label"], observed=True)
    .agg(
        total_coverage=("total_coverage", "sum"),
        political_corruption_articles=("political_corruption_articles", "sum"),
    )
    .reset_index()
)
country_summary["relative_attention"] = (
    country_summary["political_corruption_articles"] / country_summary["total_coverage"]
)
country_summary["relative_attention_pct"] = country_summary["relative_attention"] * 100
country_summary = country_summary.sort_values("relative_attention_pct", ascending=False)

overall_total = country_summary["total_coverage"].sum()
overall_pc = country_summary["political_corruption_articles"].sum()
overall_rate = overall_pc / overall_total

print(f"Total cleaned corruption-query articles: {overall_total:,}")
print(f"Total political-corruption articles:     {overall_pc:,}")
print(f"Overall relative attention:              {overall_rate:.2%}")

country_summary.to_csv(TABLE_DIR / "political_corruption_attention_country_summary.csv", index=False)
display(country_summary)

## 6. Plot Helpers

In [ ]:
COUNTRY_COLORS = {
    "Bulgaria": "#4E79A7",
    "France": "#F28E2B",
    "Hungary": "#E15759",
    "Italy": "#76B7B2",
    "Netherlands": "#59A14F",
    "Serbia": "#EDC948",
    "Sweden": "#B07AA1",
    "Ukraine": "#FF9DA7",
    "United_Kingdom": "#9C755F",
}

def period_data(level=PLOT_LEVEL):
    if level == "week":
        data = attention_week.copy()
        period = "week"
        label = "Weekly"
    elif level == "month":
        data = attention_month.copy()
        period = "month"
        label = "Monthly"
    else:
        raise ValueError("level must be 'week' or 'month'")
    return data, period, label

def save_figure(fig, name):
    png = FIGURE_DIR / f"{name}.png"
    pdf = FIGURE_DIR / f"{name}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved {png}")
    print(f"Saved {pdf}")

def format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=0)
    return ax

## 7. Relative Attention Over Time By Country

In [ ]:
data, period, level_label = period_data(PLOT_LEVEL)

fig, ax = plt.subplots(figsize=(12, 6))

for country in COUNTRY_ORDER:
    country_data = data[data["country"].astype(str).eq(country)].sort_values(period)
    if country_data.empty:
        continue
    ax.plot(
        country_data[period],
        country_data["relative_attention_pct"],
        label=COUNTRY_LABELS.get(country, country),
        color=COUNTRY_COLORS[country],
        linewidth=1.8,
        alpha=0.9,
    )

ax.set_title(f"{level_label} Relative Attention To Political Corruption By Country")
ax.set_ylabel("Political-corruption articles (% of cleaned corruption-query corpus)")
ax.set_xlabel("")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_ylim(bottom=0)
fig.tight_layout()
save_figure(fig, f"political_corruption_relative_attention_{period}_country_lines")

## 8. Small Multiples

In [ ]:
data, period, level_label = period_data(PLOT_LEVEL)

fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, country in zip(axes, COUNTRY_ORDER):
    country_data = data[data["country"].astype(str).eq(country)].sort_values(period)
    ax.plot(
        country_data[period],
        country_data["relative_attention_pct"],
        color=COUNTRY_COLORS[country],
        linewidth=1.8,
    )
    ax.fill_between(
        country_data[period],
        country_data["relative_attention_pct"],
        color=COUNTRY_COLORS[country],
        alpha=0.16,
    )
    ax.set_title(COUNTRY_LABELS.get(country, country), loc="left", fontweight="bold")
    format_time_axis(ax)
    ax.set_ylim(bottom=0)

fig.suptitle(f"{level_label} Relative Attention To Political Corruption", y=1.02, fontsize=14)
fig.text(0.5, -0.01, "Year", ha="center")
fig.text(0.0, 0.5, "% of cleaned corruption-query corpus", va="center", rotation="vertical")
fig.tight_layout()
save_figure(fig, f"political_corruption_relative_attention_{period}_small_multiples")

## 9. Absolute Political-Corruption Volume

In [ ]:
data, period, level_label = period_data(PLOT_LEVEL)

wide_counts = (
    data.pivot_table(
        index=period,
        columns="country",
        values="political_corruption_articles",
        aggfunc="sum",
        fill_value=0,
        observed=True,
    )
    .reindex(columns=COUNTRY_ORDER)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(
    wide_counts.index,
    [wide_counts[country].to_numpy() for country in COUNTRY_ORDER],
    labels=[COUNTRY_LABELS.get(country, country) for country in COUNTRY_ORDER],
    colors=[COUNTRY_COLORS[country] for country in COUNTRY_ORDER],
    alpha=0.88,
)
ax.set_title(f"{level_label} Volume Of Political-Corruption Coverage")
ax.set_ylabel("Predicted political-corruption articles")
ax.set_xlabel("")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
save_figure(fig, f"political_corruption_absolute_volume_{period}_stacked")

## 10. Country-Year Heatmap

In [ ]:
yearly_attention = attention_month.copy()
yearly_attention["year"] = yearly_attention["month"].dt.year
yearly_attention = (
    yearly_attention.groupby(["country", "country_label", "year"], observed=True)
    .agg(
        total_coverage=("total_coverage", "sum"),
        political_corruption_articles=("political_corruption_articles", "sum"),
    )
    .reset_index()
)
yearly_attention["relative_attention_pct"] = (
    yearly_attention["political_corruption_articles"] / yearly_attention["total_coverage"] * 100
)

heatmap_data = yearly_attention.pivot(index="country_label", columns="year", values="relative_attention_pct")
heatmap_data = heatmap_data.reindex([COUNTRY_LABELS.get(country, country) for country in COUNTRY_ORDER])

fig, ax = plt.subplots(figsize=(11, 5.5))
image = ax.imshow(heatmap_data, aspect="auto", cmap="YlOrRd")
ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns.astype(int))
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title("Relative Attention To Political Corruption By Country-Year")

for y in range(heatmap_data.shape[0]):
    for x in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[y, x]
        if pd.notna(value):
            ax.text(x, y, f"{value:.0f}", ha="center", va="center", fontsize=7, color="black")

cbar = fig.colorbar(image, ax=ax)
cbar.set_label("Political-corruption articles (% of cleaned corruption-query corpus)")
fig.tight_layout()
save_figure(fig, "political_corruption_relative_attention_country_year_heatmap")

yearly_attention.to_csv(TABLE_DIR / "political_corruption_attention_country_year.csv", index=False)
display(yearly_attention.head())

## 11. What To Report

For a manuscript figure, start with the monthly small-multiple relative-attention plot. Use the stacked absolute-volume plot as a secondary/descriptive figure, and the country-year heatmap as an appendix-friendly overview.